## Fase 1: Ingesta y Simulación del Sistema Origen (Raw Data con Anomalías)

**Objetivo:** Establecer el punto de partida del flujo de datos simulando la extracción desde un sistema transaccional e inyectando errores conocidos.

En esta fase inicial, configuramos la conexión a la base de datos local (`jardineria.db`) aplicando un patrón **idempotente** (eliminando y recreando estructuras).

La clave de este paso es la inyección de un lote de **datos crudos altamente contaminados**. De manera intencional, hemos introducido violaciones severas a las reglas de negocio, tales como:
* Llaves primarias duplicadas (Unicidad).
* Precios y cantidades negativas (Validez).
* Fechas de transacciones en el futuro (Validez).
* Ventas asociadas a clientes fantasma (Integridad Referencial).

Este conjunto de datos "sucios" es fundamental para someter a estrés nuestro Data Quality Dashboard en las fases posteriores y certificar que el motor detecta errores críticos.

In [17]:
# ==============================================================================
# PASO 1: CREACIÓN DE BASE DE DATOS ORIGEN Y CARGA DE DATOS SUCIOS
# ==============================================================================
import sqlite3
import pandas as pd

# Conectamos/Creamos la base de datos
conn = sqlite3.connect('jardineria.db')
cursor = conn.cursor()

# Limpiamos y creamos las tablas originales
cursor.executescript('''
DROP TABLE IF EXISTS cliente;
DROP TABLE IF EXISTS empleado;
DROP TABLE IF EXISTS oficina;
DROP TABLE IF EXISTS pedido;
DROP TABLE IF EXISTS detalle_pedido;
DROP TABLE IF EXISTS producto;

CREATE TABLE cliente (ID_cliente INT, nombre_cliente TEXT, ciudad TEXT, pais TEXT, region TEXT, ID_empleado_rep_ventas INT);
CREATE TABLE empleado (ID_empleado INT, ID_oficina TEXT);
CREATE TABLE oficina (ID_oficina TEXT, Descripcion TEXT, ciudad TEXT, pais TEXT);
CREATE TABLE pedido (ID_pedido INT, fecha_pedido DATE, estado TEXT, ID_cliente INT);
CREATE TABLE detalle_pedido (ID_detalle_pedido INT, ID_pedido INT, ID_producto TEXT, cantidad INT, precio_unidad REAL);
CREATE TABLE producto (ID_producto TEXT, nombre TEXT, Categoria INT, precio_proveedor REAL);

-- ==========================================
-- INYECCIÓN DE ANOMALÍAS (DATA POISONING)
-- ==========================================

-- 1. Dimensión Clientes
INSERT INTO cliente VALUES (1, 'Tienda A', 'Bogotá', 'Colombia', 'Cundinamarca', 10); -- OK
INSERT INTO cliente VALUES (2, 'Tienda B', 'Medellín', 'Colombia', 'Antioquia', 99); -- ADVERTENCIA: Empleado 99 no existe (Sin Oficina)
INSERT INTO cliente VALUES (2, 'Tienda B Clon', 'Medellín', 'Colombia', 'Antioquia', 10); -- CRÍTICO: ID_cliente duplicado
INSERT INTO cliente VALUES (3, 'Tienda C', 'Cali', 'Colombia', NULL, 10); -- ADVERTENCIA: Región nula

-- 2. Dimensiones Empleado, Oficina y Producto
INSERT INTO empleado VALUES (10, 'OF-COL');
INSERT INTO oficina VALUES ('OF-COL', 'Oficina Central', 'Bogotá', 'Colombia');
INSERT INTO producto VALUES ('PROD-01', 'Pala', 1, 50.0);

-- 3. Tabla de Hechos: Pedidos
INSERT INTO pedido VALUES (100, '2026-03-29', 'Entregado', 1); -- OK
INSERT INTO pedido VALUES (101, '2030-12-31', 'Extraviado', 1); -- CRÍTICO/ADVERTENCIA: Fecha futura y Estado no homologado
INSERT INTO pedido VALUES (102, '2026-03-25', 'Pendiente', 999); -- CRÍTICO: Cliente 999 no existe (Huérfano)

-- 4. Tabla de Hechos: Detalle de Pedidos
INSERT INTO detalle_pedido VALUES (500, 100, 'PROD-01', 10, 60.0); -- OK
INSERT INTO detalle_pedido VALUES (500, 100, 'PROD-01', 10, 60.0); -- CRÍTICO: ID_detalle_pedido duplicado
INSERT INTO detalle_pedido VALUES (501, 101, 'PROD-01', -5, -15.0); -- CRÍTICO: Cantidad y Precio negativos
''')

print("¡Paso 1 completado! Base de datos origen creada e inyectada con errores críticos.")

¡Paso 1 completado! Base de datos origen creada e inyectada con errores críticos.


## Fase 2: Capa de Staging y Transformación Analítica (ETL)

**Objetivo:** Consolidar, denormalizar y enriquecer los datos crudos preparándolos para la evaluación de calidad y el análisis posterior.

En esta fase, implementamos el proceso de transformación utilizando el patrón **CTAS (Create Table As Select)**, el cual es altamente eficiente para cargas analíticas. El script ejecuta dos operaciones arquitectónicas críticas:

1. **Denormalización de Dimensiones:** Unifica las entidades de `cliente`, `empleado` y `oficina` en una sola tabla plana (`stg_clientes_geografia`) mediante el uso de `LEFT JOIN`. Esto facilita el análisis geográfico downstream y expone posibles huérfanos relacionales.
2. **Consolidación de Hechos y Reglas de Negocio:** Centraliza la información transaccional de ventas (`stg_ventas_detalle`) integrando pedidos, detalles y productos. Además, **pre-calcula métricas financieras clave al vuelo** (como `subtotal` y `margen_ganancia`), asegurando que la lógica de negocio quede estandarizada y centralizada antes de llegar a la capa de reportes.

Mantenemos el enfoque idempotente (`DROP TABLE IF EXISTS`) asegurando que el área de Staging se refresque limpiamente en cada ejecución.

In [18]:
cursor.executescript('''

-- 1. Limpiamos el Staging de Clientes
DROP TABLE IF EXISTS stg_clientes_geografia;

-- Creamos y poblamos la tabla en una sola operación
CREATE TABLE stg_clientes_geografia AS
SELECT
c.ID_cliente,
c.nombre_cliente,
c.ciudad AS ciudad_cliente,
c.pais AS pais_cliente,
c.region AS region_cliente,
o.Descripcion AS nombre_oficina,
o.ciudad AS ciudad_oficina,
o.pais AS pais_oficina

FROM cliente c
LEFT JOIN empleado e ON c.ID_empleado_rep_ventas = e.ID_empleado
LEFT JOIN oficina o ON e.ID_oficina = o.ID_oficina;

-- 2. Limpiamos el Staging de Ventas
DROP TABLE IF EXISTS stg_ventas_detalle;

-- Creamos y poblamos la tabla de ventas con sus métricas calculadas
CREATE TABLE stg_ventas_detalle AS
SELECT
dp.ID_detalle_pedido,
p.ID_pedido,
p.fecha_pedido,
p.estado AS estado_pedido,
p.ID_cliente,
pr.nombre AS nombre_producto,
pr.Categoria AS categoria_producto,
dp.cantidad,
dp.precio_unidad,
(dp.cantidad * dp.precio_unidad) AS subtotal,
((dp.precio_unidad - pr.precio_proveedor) * dp.cantidad) AS margen_ganancia
FROM pedido p
INNER JOIN detalle_pedido dp ON p.ID_pedido = dp.ID_pedido
INNER JOIN producto pr ON dp.ID_producto = pr.ID_producto;
''')

print("¡Éxito! Tablas de Staging procesadas usando el patrón CTAS.")

¡Éxito! Tablas de Staging procesadas usando el patrón CTAS.


## Fase 3: Auditoría de Calidad de Datos (Data Quality Dashboard)

**Objetivo:** Evaluar la salud de los datos en la capa de Staging mediante un motor de validación automatizado basado en las dimensiones estándar de calidad de datos.

En esta fase crítica, implementamos un *framework* de pruebas estructurado que audita el 100% de los registros procesados. El código se divide en tres componentes lógicos:

1. **Diccionario de Reglas (DQ Rules):** Traducción de las reglas de negocio a consultas SQL orientadas a buscar fallas. Cubrimos dimensiones clave del framework DAMA: **Completitud** (nulos), **Unicidad** (duplicados), **Integridad Referencial** (huérfanos), **Validez** (rangos lógicos), **Exactitud** (cálculos financieros) y **Consistencia** (homologación de estados).
2. **Motor de Ejecución y Triage:** Un bucle iterativo que ejecuta las pruebas y clasifica la severidad del fallo (Crítico o Advertencia) dependiendo del impacto potencial en el negocio.
3. **Renderizado Visual:** Transformación de los resultados métricos en un *dashboard* tabular utilizando Pandas Styling. Se implementa un semáforo visual que permite a los *Stakeholders* o Data Stewards identificar anomalías de un solo vistazo.

In [25]:
import pandas as pd

# 1. DEFINICIÓN DE REGLAS DE CALIDAD
queries_calidad = {
    # --- COMPLETITUD ---
    "Clientes sin Oficina":
        "SELECT COUNT(*) FROM stg_clientes_geografia WHERE nombre_oficina IS NULL",

    # --- UNICIDAD ---
    "IDs de Cliente Duplicados":
        "SELECT COUNT(*) FROM (SELECT ID_cliente FROM stg_clientes_geografia GROUP BY ID_cliente HAVING COUNT(*) > 1)",

    "Detalles de Pedido Duplicados":
        "SELECT COUNT(*) FROM (SELECT ID_detalle_pedido FROM stg_ventas_detalle GROUP BY ID_detalle_pedido HAVING COUNT(*) > 1)",

    # --- INTEGRIDAD REFERENCIAL ---
    "Ventas de Clientes Inexistentes":
        "SELECT COUNT(*) FROM stg_ventas_detalle WHERE ID_cliente NOT IN (SELECT ID_cliente FROM stg_clientes_geografia)",

    # --- VALIDEZ (RANGOS Y FORMATOS) ---
    r"Cantidades de Producto $\leq 0$":
        "SELECT COUNT(*) FROM stg_ventas_detalle WHERE cantidad <= 0",

    "Fechas de Pedido en el Futuro":
        "SELECT COUNT(*) FROM stg_ventas_detalle WHERE fecha_pedido > date('now')",

    r"Precios Unidad $\leq 0$":
        "SELECT COUNT(*) FROM stg_ventas_detalle WHERE precio_unidad <= 0",

    # --- EXACTITUD (CÁLCULOS LOGICOS) ---
    "Error en Cálculo de Subtotal":
        "SELECT COUNT(*) FROM stg_ventas_detalle WHERE ABS(subtotal - (cantidad * precio_unidad)) > 0.01",

    "Error en Cálculo de Margen (Nulos)":
        "SELECT COUNT(*) FROM stg_ventas_detalle WHERE margen_ganancia IS NULL",

    # --- CONSISTENCIA ---
    "Clientes sin Región Definida":
        "SELECT COUNT(*) FROM stg_clientes_geografia WHERE region_cliente IS NULL OR region_cliente = ''",

    "Estados de Pedido no Homologados":
        "SELECT COUNT(*) FROM stg_ventas_detalle WHERE estado_pedido NOT IN ('Entregado', 'Pendiente', 'Rechazado', 'Enviado')"
}

# 2. MOTOR DE EJECUCIÓN
resultados = []

for nombre, query in queries_calidad.items():
    try:
        # Ejecutamos la consulta contra nuestra base de datos en memoria
        res = cursor.execute(query).fetchone()[0]

        # Asignamos el semáforo de criticidad
        if res == 0:
            estado = "✅ PASA"
        elif any(x in nombre for x in ["Duplicados", "Inexistentes", r"$\leq 0$"]):
            estado = "🚨 CRÍTICO"
        else:
            estado = "⚠️ ADVERTENCIA"

        resultados.append({"Prueba de Calidad": nombre, "Registros Fallidos": res, "Estatus": estado})
    except Exception as e:
        resultados.append({"Prueba de Calidad": nombre, "Registros Fallidos": "Error", "Estatus": f"❌ {str(e)}"})

# 3. RENDERIZADO DEL DASHBOARD
df_reporte = pd.DataFrame(resultados)

# Aplicamos estilos visuales al DataFrame
def highlight_status(val):
    if val == '✅ PASA': return 'color: #2ecc71; font-weight: bold'
    if '🚨 CRÍTICO' in val: return 'color: #e74c3c; font-weight: bold'
    if '⚠️' in val: return 'color: #f1c40f; font-weight: bold'
    return ''

# Mostrar la tabla estilizada
display(df_reporte.style.map(highlight_status, subset=['Estatus']))

,Prueba de Calidad,Registros Fallidos,Estatus
0,Clientes sin Oficina,0,✅ PASA
1,IDs de Cliente Duplicados,0,✅ PASA
2,Detalles de Pedido Duplicados,0,✅ PASA
3,Ventas de Clientes Inexistentes,0,✅ PASA
4,Cantidades de Producto $\leq 0$,0,✅ PASA
5,Fechas de Pedido en el Futuro,0,✅ PASA
6,Precios Unidad $\leq 0$,0,✅ PASA
7,Error en Cálculo de Subtotal,0,✅ PASA
8,Error en Cálculo de Margen (Nulos),0,✅ PASA
9,Clientes sin Región Definida,0,✅ PASA


## Fase 4: Compuerta de Calidad (Data Quality Gate & Circuit Breaker)

**Objetivo:** Interrumpir el flujo de ejecución del *pipeline* (Hard Fail) si se detectan anomalías de severidad crítica que comprometan la integridad del modelo de datos.

En arquitecturas de datos robustas, no basta con monitorear pasivamente los errores. Mientras que las advertencias (`⚠️ ADVERTENCIA`) pueden ser mitigadas mediante reglas de imputación, los errores estructurales (`🚨 CRÍTICO`) representan violaciones directas a los **Contratos de Datos** (Data Contracts).

En esta fase, implementamos un *Circuit Breaker* (Interruptor de Circuito). Este script evalúa los resultados del motor de calidad anterior; si el conteo de fallos críticos es mayor a cero, el sistema levanta una excepción (`ValueError`). Esto detiene inmediatamente la ejecución del *notebook* (o del orquestador, como Airflow), protegiendo la capa final (Data Warehouse) de la propagación de datos corruptos.

In [23]:
# ==============================================================================
# PASO 4: COMPUERTA DE CALIDAD (CIRCUIT BREAKER)
# ==============================================================================

print("🛡️ Evaluando la Compuerta de Calidad (Data Quality Gate)...\n")

# 1. Filtramos del reporte anterior solo las pruebas que resultaron en "CRÍTICO"
df_criticos = df_reporte[df_reporte['Estatus'] == '🚨 CRÍTICO']

# 2. Sumamos el total de registros fallidos en esas pruebas críticas
total_errores_criticos = pd.to_numeric(df_criticos['Registros Fallidos'], errors='coerce').sum()

# 3. Lógica del Circuit Breaker (Hard Fail vs Pass)
if total_errores_criticos > 0:
    # Extraemos los nombres de las pruebas específicas que detonaron la alarma
    pruebas_fallidas = df_criticos[df_criticos['Registros Fallidos'] > 0]['Prueba de Calidad'].tolist()

    mensaje_error = (
        f"\n❌ FATAL ERROR: El pipeline ha sido detenido preventivamente.\n"
        f"Se encontraron {int(total_errores_criticos)} anomalías con severidad CRÍTICA.\n"
        f"Pruebas que fallaron: {pruebas_fallidas}\n\n"
        f"ACCIÓN REQUERIDA: Revise los datos de origen (Ingesta) o aplique reglas de "
        f"cuarentena antes de permitir que los datos avancen al Data Warehouse."
    )

    # Levantamos una excepción real que detiene la ejecución de Colab
    raise ValueError(mensaje_error)

else:
    print("✅ COMPUERTA SUPERADA: No hay errores críticos estructurales.")
    print("El pipeline está autorizado para continuar a la fase de Limpieza y Producción.")

🛡️ Evaluando la Compuerta de Calidad (Data Quality Gate)...

✅ COMPUERTA SUPERADA: No hay errores críticos estructurales.
El pipeline está autorizado para continuar a la fase de Limpieza y Producción.


## Fase 5: Remediación Avanzada (Deduplicación y Cuarentena de Datos)

**Objetivo:** Aplicar transformaciones correctivas automatizadas para sanear las anomalías críticas detectadas, permitiendo que el *pipeline* supere el *Circuit Breaker*.

Cuando un *pipeline* sufre un *Hard Fail* debido a violaciones de los Contratos de Datos, es imperativo aplicar estrategias de limpieza programática sin perder la trazabilidad de los errores. En esta fase implementamos dos patrones de resiliencia:

1. **Patrón de Cuarentena (Dead Letter Queue / Quarantine):** Los registros transaccionales con errores financieros graves (cantidades o precios $\leq 0$) no se eliminan físicamente; se aíslan en una tabla de cuarentena (`stg_ventas_cuarentena`). Esto preserva la evidencia para auditoría y sanea la tabla principal.
2. **Deduplicación (Record Linkage / Survivorship):** Eliminación de tuplas clonadas en las dimensiones de clientes y en la tabla de hechos de ventas, garantizando la **Unicidad** mediante la retención del primer registro insertado.

In [24]:
# ==============================================================================
# PASO 5: CUARENTENA Y DEDUPLICACIÓN (Sanitización Crítica)
# ==============================================================================

print("🛠️ Iniciando proceso de remediación avanzada...")

cursor.executescript('''
-- ==========================================
-- 1. PATRÓN DE CUARENTENA (Data Quarantine)
-- ==========================================
-- Creamos la tabla de cuarentena con la misma estructura que ventas
DROP TABLE IF EXISTS stg_ventas_cuarentena;
CREATE TABLE stg_ventas_cuarentena AS
SELECT * FROM stg_ventas_detalle WHERE 1=0; -- Copia solo el esquema vacío

-- Movemos los registros "tóxicos" (precios/cantidades negativas) a cuarentena
INSERT INTO stg_ventas_cuarentena
SELECT * FROM stg_ventas_detalle
WHERE cantidad <= 0 OR precio_unidad <= 0;

-- Eliminamos los registros tóxicos de la tabla principal
DELETE FROM stg_ventas_detalle
WHERE cantidad <= 0 OR precio_unidad <= 0;

-- ==========================================
-- 2. DEDUPLICACIÓN DE DATOS (Survivorship)
-- ==========================================
-- Eliminamos clientes duplicados conservando solo el primer ROWID
DELETE FROM stg_clientes_geografia
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM stg_clientes_geografia
    GROUP BY ID_cliente
);

-- Eliminamos detalles de pedido duplicados conservando el primer ROWID
DELETE FROM stg_ventas_detalle
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM stg_ventas_detalle
    GROUP BY ID_detalle_pedido
);

-- ==========================================
-- 3. LIMPIEZA DE ADVERTENCIAS (Del Paso Anterior)
-- ==========================================
-- Imputamos la oficina faltante y la región nula para limpiar todo el dashboard
UPDATE stg_clientes_geografia
SET nombre_oficina = 'Oficina por Defecto', ciudad_oficina = 'N/A', pais_oficina = 'N/A'
WHERE nombre_oficina IS NULL;

UPDATE stg_clientes_geografia
SET region_cliente = 'No Definida'
WHERE region_cliente IS NULL;
''')

# Validamos cuántos registros enviamos a cuarentena
cuarentena_count = cursor.execute("SELECT COUNT(*) FROM stg_ventas_cuarentena").fetchone()[0]
print(f"✅ Sanitización completada. Registros enviados a cuarentena: {cuarentena_count}")
print("El Staging ha sido limpiado. Puede reevaluar la compuerta de calidad.")

🛠️ Iniciando proceso de remediación avanzada...
✅ Sanitización completada. Registros enviados a cuarentena: 0
El Staging ha sido limpiado. Puede reevaluar la compuerta de calidad.


## Fase Final: Validación y Monitoreo Continuo (Post-Cleansing Verification)

**Objetivo:** Confirmar la efectividad de las reglas de remediación y garantizar que el dataset cumpla al 100% con los estándares de calidad antes de su liberación a producción.

En la arquitectura de datos, ninguna transformación se considera exitosa hasta que es empíricamente validada. En esta etapa de cierre, procedemos a re-ejecutar el **Data Quality Dashboard** (nuestro motor de reglas de calidad definido en el Paso 3).

El propósito de esta re-evaluación iterativa es doble:
1. **Certificación de Limpieza:** Comprobar que las alertas previas (como los huérfanos geográficos) hayan sido neutralizadas y su estatus haya cambiado de `⚠️ ADVERTENCIA` a `✅ PASA`.
2. **Prueba de Regresión:** Asegurar que las sentencias de actualización (`UPDATE`) aplicadas en la fase de curación no hayan introducido efectos secundarios o corrompido otras dimensiones del *dataset*.

Este ciclo riguroso de *Auditoría $\rightarrow$ Limpieza $\rightarrow$ Re-Auditoría* es el estándar de oro metodológico para mantener una estricta gobernanza, asegurando que los analistas de negocio consuman información completamente confiable.